In [10]:
import arxiv
import os
import logging
import asyncio
import os
import dotenv
import ollama
# Import for SerpAPI 1.5.0
from serpapi.google_search import GoogleSearch
# To install: pip install tavily-python
from tavily import TavilyClient

dotenv.load_dotenv()

SERP_API_KEY = os.getenv("SERP_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
QUIVERQUANT_API_KEY = os.getenv("QUIVERQUANT_API_KEY")


def arxiv_search(query: str, max_results: int = 5) -> list[dict]:
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )
    results = []
    for result in search.results():
        results.append({
            "title": result.title,
            "authors": [author.name for author in result.authors],
            "summary": result.summary,
            "pdf_url": result.pdf_url
        })
    return results

# Tool: Search Google using SerpApi
def serp_search(query: str) -> list[dict]:
    if not SERP_API_KEY:
        return [{"error": "API key not configured"}]
    
    params = {
        "q": query,
        "api_key": SERP_API_KEY,
        "num": 5
    }
    
    try:
        search = GoogleSearch(params)
        results = search.get_dict()
        return results.get("organic_results", [])
    except Exception as e:
        return [{"error": str(e)}]
    
# Tool: Search Google using Tavily
def tavily_search(query: str) -> list[dict]:
    if not TAVILY_API_KEY:
        return [{"error": "API key not configured"}]

    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(query=query, query_results=5)
    return response.get("results", [])

async def test_ollama(model: str = "llama3.1:8b", host: str = "http://localhost:11434", prompt: str = "What is a mango?"):
    try:
        client = ollama.AsyncClient(host=host)
    except Exception as e:
        logging.error(f"Failed to create Olama client at {host}: {e}")
        return {"error": str(e)}
    try:
        response = await client.chat(
            model=model,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
    except Exception as e:
        logging.error(f"Failed to get response from Ollama server at {host}: {e}")
        return {"error": str(e)}
    return response

In [ ]:
# Validated test cases
response = arxiv_search(
    query="LLM agent security AWS",
    max_results=5
)
print(f"ARXIV TEST: {response}")

response = tavily_search(
    query="Operation Beluga"
)
print(f"TAVILY TEST: {response}")

# To run the async test function
try:
    ollama_results = await test_ollama(model="llama3.1:8b")
    print(f"OLLAMA TEST: {ollama_results}")
except Exception as e:
    print(f"An error occurred while running the Ollama test: {e}")

In [21]:
# /api/daily_papers
import requests
import json
response = requests.get("https://huggingface.co/api/daily_papers")

In [27]:
loaded_content = json.loads(response.content)
for paper in loaded_content:
    print(f"Title: {paper['title']}")
    print(f"Summary: {paper['summary']}")
    print("-" * 40)


Title: RiemannLoRA: A Unified Riemannian Framework for Ambiguity-Free LoRA
  Optimization
Summary: Low-Rank Adaptation (LoRA) has become a widely adopted standard for
parameter-efficient fine-tuning of large language models (LLMs), significantly
reducing memory and computational demands. However, challenges remain,
including finding optimal initialization strategies or mitigating
overparametrization in low-rank matrix factorization. In this work, we propose
a novel approach that addresses both of the challenges simultaneously within a
unified framework. Our method treats a set of fixed-rank LoRA matrices as a
smooth manifold. Considering adapters as elements on this manifold removes
overparametrization, while determining the direction of the fastest loss
decrease along the manifold provides initialization. Special care is taken to
obtain numerically stable and computationally efficient implementation of our
method, using best practices from numerical linear algebra and Riemannian
optim